# Mixture of Experts (toy)

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

A MoE layer replaces the dense FFN by $E$ smaller "expert" FFNs and a *gating* network that, for each token, picks the top-$k$ experts to run. Capacity scales with $E$ but compute scales with $k \ll E$.


## Mathematical Formulation

$$g = \text{softmax}(W_g x)$$
$$\text{topk-experts}(x) = \sum_{i \in \text{TopK}(g)} \frac{g_i}{\sum_j g_j} \cdot E_i(x)$$

An auxiliary load-balance loss keeps experts from collapsing onto a few favourites.


## Implementation


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class MoE(nn.Module):
    def __init__(self, d_model, n_experts=4, top_k=2, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
            for _ in range(n_experts)
        ])
        self.gate = nn.Linear(d_model, n_experts)
        self.top_k = top_k

    def forward(self, x):
        B, N, D = x.shape
        flat = x.reshape(B * N, D)
        logits = self.gate(flat)                      # (BN, E)
        scores, idx = logits.topk(self.top_k, dim=-1)
        weights = F.softmax(scores, dim=-1)            # (BN, k)
        out = torch.zeros_like(flat)
        for k in range(self.top_k):
            for e, expert in enumerate(self.experts):
                mask = idx[:, k] == e
                if mask.any():
                    out[mask] += weights[mask, k:k+1] * expert(flat[mask])
        # load-balance auxiliary loss
        probs = F.softmax(logits, dim=-1).mean(0)
        usage = (idx == torch.arange(len(self.experts), device=x.device)).float().mean((0, 1))
        aux = (probs * usage).sum() * len(self.experts)
        return out.reshape(B, N, D), aux


## Experiment


In [ ]:
moe = MoE(d_model=64, n_experts=4, top_k=2)
x = torch.randn(2, 16, 64)
y, aux = moe(x)
print('output:', y.shape, 'aux loss:', aux.item())


In [ ]:
# Train briefly with the auxiliary loss to see balance improve
opt = torch.optim.AdamW(moe.parameters(), lr=1e-3)
for step in range(50):
    y, aux = moe(x)
    loss = y.pow(2).mean() + 0.01 * aux
    opt.zero_grad(); loss.backward(); opt.step()
print('final aux loss:', aux.item())


## Discussion

- The "for-loop over experts" implementation is the simplest possible; real systems use grouped GEMMs or expert-parallel kernels.
- Routing is *not* differentiable through the TopK selection — backprop only flows through the active experts.
- Mixtral-style MoE uses $E=8, k=2$ per token and ties experts across layers via sparse routing.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
